<a href="https://www.kaggle.com/code/ps6nlb/267600821-analyze-sentiment?scriptVersionId=329402478" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
!pip install pythainlp

# Import Library

In [ ]:
import numpy as np 
import pandas as pd 
from pythainlp.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import recall_score, precision_score, accuracy_score, classification_report

# สำรวจข้อมูล Wongnai-Food-Review-Rating-Prediction-Challenge

In [ ]:
df = pd.read_csv('/kaggle/input/competitions/wongnai-food-review-rating-prediction-challenge/train_data.csv')
df

In [ ]:
df.groupby('star_rating').count()

# การจัดเตรียมข้อมูล (Data Preparation)

แปลงตัวแปรเป้าหมาย (Label) จาก star_rating เป็นค่าประเมินความคิดเห็นตามโจทย์ <br> Start_Rating 0-1 แปลงเป็นความคิดเห็นเชิงลบ (neg : 0) <br> Start_Rating 2 แปลงเป็นความคิดเห็นทั่วไป (gen : 1) <br>Start_Rating 3-4 แปลงเป็นความคิดเห็นเชิงบวก (pos : 2)

In [ ]:
df_neg = df[(df['star_rating'] == 0) | (df['star_rating'] == 1) ]
df_neg

In [ ]:
df_neg.loc[:, ['star_rating']] = 0
df_neg

In [ ]:
df_gen = df[df['star_rating'] == 2]
df_gen

In [ ]:
df_gen.loc[:, ['star_rating']] = 1
df_gen

In [ ]:
df_pos = df[(df['star_rating'] == 3) | (df['star_rating'] == 4) ]
df_pos.loc[:, ['star_rating']] = 2
df_pos

In [ ]:
df1 = pd.concat([df_neg, df_gen, df_pos])
df1

แบ่งข้อมูล Train Set/Test Set ในอัตราส่วน 80:20 โดยวิธีการสุ่ม (Sample)

In [ ]:
df_train = df1.sample(32000)
df_train

In [ ]:
df_test = df1.drop(df_train.index)
df_test

แยกคำที่มีในพจนานุกรมภาษาไทยลงใน List โดยใช้ไลบรารี่ pythainlp

In [ ]:
def token_thai(text):
    tokens = word_tokenize(text, engine='newmm')
    return [word for word in tokens if word.strip() != '']

In [ ]:
token_thai('ทดสอบแยกคำในภาษาไทย')

นำ Data Set ไปวิเคราะห์ตาราง TF-IDF เพื่อหาคำเพื่อปราฏบ่อยที่สุดในเอกสาร

In [ ]:
vectorizer = TfidfVectorizer(tokenizer=token_thai, token_pattern=None, max_features=10000, min_df= 5, max_df=0.9)
x_train = vectorizer.fit_transform(df_train['review_body'])

In [ ]:
y_train = df_train['star_rating']
y_train

In [ ]:
x_test = vectorizer.transform(df_test['review_body'])
x_test

In [ ]:
y_test = df_test['star_rating']
y_test

# การประมวลผลการพยากรณ์โดยใช้ Machine Learning / การประเมินโมเดล Classification

In [ ]:
model_nai = MultinomialNB()
model_nai.fit(x_train, y_train)
y_pred = model_nai.predict(x_test)
print(classification_report(y_test,y_pred))

In [ ]:
model_svc = LinearSVC()
model_svc.fit(x_train, y_train)
y_pred = model_svc.predict(x_test)
print(classification_report(y_test,y_pred))

In [ ]:
model_dt =  DecisionTreeClassifier()
model_dt.fit(x_train, y_train)
y_pred = model_dt.predict(x_test)
print(classification_report(y_test,y_pred))

เลือกใช้ Model ที่มีค่าผลการประเมินที่ดีที่สุดนั้นคือโมเดล LogisticRegression Classification

In [ ]:
model_lr = LogisticRegression()
model_lr.fit(x_train, y_train)
y_pred = model_lr.predict(x_test)
print(classification_report(y_test,y_pred))

In [ ]:
y_pred

หา Feature Impotant จาก Model ที่ดีที่สุด เพื่อหาคำที่มีผลต่อการทำนายของโมเดลมากที่สุด โดยแบ่งเป็น 3 คลาส ได้แก่ ความคิดเห็นเชิงลบ (neg), ความคิดเห็นทั่วไป (gen), ความคิดเห็นเชิงบวก (pos)

In [ ]:
df_feature_neg = pd.DataFrame({'Feature' : np.array(vectorizer.get_feature_names_out()),'Coef' : np.array(model_lr.coef_[0])})
df_feature_neg.sort_values(by = 'Coef', ascending=False)[0:9]

In [ ]:
df_feature_gen = pd.DataFrame({'Feature' : np.array(vectorizer.get_feature_names_out()),'Coef' : np.array(model_lr.coef_[1])})
df_feature_gen.sort_values(by = 'Coef', ascending=False)[0:9]

In [ ]:
df_feature_pos = pd.DataFrame({'Feature' : np.array(vectorizer.get_feature_names_out()),'Coef' : np.array(model_lr.coef_[2])})
df_feature_pos.sort_values(by = 'Coef', ascending=False)[0:9]

# การทดสอบชุดข้อมูล Test Set

นำข้อมูลจากการทำนายโมเดลมาเปรียบเทียบกับข้อมูลจริงในชุดข้อมูล TestSet

In [ ]:
df_ex = df_test.copy()
df_ex['predict'] = y_pred
df_ex

In [ ]:
df_ex.to_csv('result1.csv', encoding='utf-8-sig')

กรองข้อมูลเฉพาะโมเดลที่ทำนายถูก

In [ ]:
df_ex_true = df_ex[df_ex['star_rating'] == df_ex['predict']]
df_ex_true

In [ ]:
df_ex_true[df_ex_true['predict'] == 2]

กรองข้อมูลเฉพาะโมเดลที่ทำนายผิด

In [ ]:
df_ex_fault = df_ex.drop(df_ex_true.index)
df_ex_fault

In [ ]:
df_ex_fault[df_ex_fault['predict'] == 1]

# การประยุกต์นำเอาไปใช้งาน โดยการสร้างฟังก์ชัน analyze_sentiment (sentence) และทดสอบข้อความอื่นๆ ที่ไม่มีใน Dataset

In [ ]:
def analyze_sentiment (sentence) :
    x_test = vectorizer.transform(pd.Series(sentence))
    y_pred = model_lr.predict(x_test)
    
    if y_pred.item() == 0:
        y_comment = 'ลบ'
    elif y_pred.item() == 1:
        y_comment = 'ทั่วไป'
    else:
        y_comment = 'บวก' 
        
    print(f'ความคิดเห็นเชิง : {y_comment}')

ทดสอบประโยคนอกเหนือจากใน Dataset 5 ตัวอย่าง

In [71]:
# Ex.1 Case: รีวิวคำชม, ความคาดหวัง : บวก
sentence = 'เนื้อย่างพรีเมียมมาก นุ่มละลายในปาก พนักงานดูแลใส่ใจดี เปลี่ยนตะแกรงให้ตลอด คุ้มค่าสมราคาครับ แนะนำเลยไม่ผิดหวังแน่นอน'
analyze_sentiment (sentence)

ความคิดเห็นเชิง : บวก


In [72]:
# EX.3 Case : รีวิวกลางๆ, ความคาดหวัง : เชิงทั่วไป
sentence = 'ร้านตกแต่งสวยดี ถ่ายรูปมุมไหนก็สวย แต่อาหารรสชาติกลางๆ ไม่ได้โดดเด่นอะไรเมื่อเทียบกับร้านอื่น ราคาแอบแรงไปนิดนึง'
analyze_sentiment (sentence)

ความคิดเห็นเชิง : ทั่วไป


In [77]:
# EX.3 Case : รีวิวไม่ดี, ความคาดหวัง : เชิงลบ
sentence = 'อาหารเค็มจัด รอนานเป็นชั่วโมงแถมได้ออเดอร์ผิด พนักงานหน้าหงิกมากตอนทวงถาม เสียดายเงินและเวลาสุดๆ ไม่มาเหยียบอีกแล้ว'
analyze_sentiment (sentence)

ความคิดเห็นเชิง : ลบ


In [76]:
# EX.4 Case การรีวิวแบบประชดประชนในบริบทคำพูดในภาษาไทย, ความคาดหวัง : เชิงลบ
sentence = 'อร่อยจนลืมกลืนเลยครับ เพราะเนื้อเหนียวเป็นยางรถยนต์ เคี้ยวจนกรามค้าง ดีจริงๆ ดีที่ฟันไม่หัก'
analyze_sentiment (sentence)

ความคิดเห็นเชิง : บวก


In [78]:
# EX.5 Case ภาษาวัยรุ่น, ความคาดหวัง : เชิงลบ
sentence = 'รสชาติคือจะเครซี่ นอยด์อ่า บ้งมาก ช็อตฟีลสุดๆ ไม่จึ้งเลยแม่ง'
analyze_sentiment (sentence)

ความคิดเห็นเชิง : บวก
